In [1]:
from pyscf import gto, fci, mcscf
import numpy as np
from gpu4pyscf import scf


/home/mert/.local/lib/python3.10/site-packages/gpu4pyscf/lib/cutensor.py:154: UserWarning: using cupy as the tensor contraction engine.
  warnings.warn(f'using {contract_engine} as the tensor contraction engine.')


In [2]:
# Molecular Information part
# !!!!! in CI part mo not forget the change electrons 
# !!!!! also orbitals are important  

xyz = '''
 C    0.000    1.391    0.000
 C    1.205    0.696    0.000
 C    1.205   -0.696    0.000
 C    0.000   -1.391    0.000
 C   -1.205   -0.696    0.000
 C   -1.205    0.696    0.000
 H    0.000    2.473    0.000
 H    2.142    1.236    0.000
 H    2.142   -1.236    0.000
 H    0.000   -2.473    0.000
 H   -2.142   -1.236    0.000
 H   -2.142    1.236    0.000'''
molecule = 'C6H6'
spin = 0
charge =  0


In [3]:
def rhf_calculation(xyz, molecule, spin, charge):
    mol = gto.M(
          atom= xyz,
          basis = 'sto-6g',
          charge = charge,
          verbose = 4 ,
          spin = spin, 
          symmetry = True,
          output  ="./"+molecule+".txt")

    mf = mol.RHF().run()
    mf.analyze() 
    S_ij = mf.get_ovlp()
    C_ij = mf.mo_coeff
    np.savetxt("C_"+molecule+'mo_coeff.txt', C_ij,fmt = '%.8f')
    np.savetxt('S_'+molecule+'_overlap.txt', S_ij,fmt = '%.8f')

    return mf, mol 

In [4]:
def orb_occ_table(orb_energy,orb_occ):
    Homo = None
    Lumo = None
    print ("orbital  energy   occ")

    for i in range (0,(len(orb_occ))):
        print(i ,"   ", orb_energy[i],"  ",orb_occ[i])
              
    for i in range(len(orb_occ) - 1):
        if orb_occ[i] == 2  and orb_occ[i+1] == 0:
            homo = i
            lumo = i + 1
            print(f"Homo_index: {homo}, Lumo_index: {lumo}")
    return homo, lumo


In [5]:
mf, mol  = rhf_calculation( xyz , molecule, spin, charge )
homo , lumo =orb_occ_table(mf.mo_energy, mf.mo_occ)

overwrite output file: ./C6H6.txt
orbital  energy   occ
0     -11.180374293001826    2.0
1     -11.180159214817024    2.0
2     -11.180141448707182    2.0
3     -11.179467874144352    2.0
4     -11.179456019353754    2.0
5     -11.179145019562817    2.0
6     -1.0934757167570763    2.0
7     -0.958768344266399    2.0
8     -0.9586217224901011    2.0
9     -0.7708765744901013    2.0
10     -0.7706595034388963    2.0
11     -0.665823143337128    2.0
12     -0.5983842088054565    2.0
13     -0.5564856657574383    2.0
14     -0.5372738407370622    2.0
15     -0.5372622347084233    2.0
16     -0.45979062705272644    2.0
17     -0.43690194653666203    2.0
18     -0.43620673132164267    2.0
19     -0.2842115769755735    2.0
20     -0.2838634956780348    2.0
21     0.2622503818509128    0.0
22     0.26274603185669687    0.0
23     0.4956878940385707    0.0
24     0.5748565992864134    0.0
25     0.642974835387968    0.0
26     0.6439910914107971    0.0
27     0.7195132592997847    0.0
28     0

In [7]:
n_elec= 6
n_orb = 6 # number of orbital can be determined based on the number of pi  orbitals 
casscf = mcscf.CASSCF(mf, n_orb, n_elec)
'''output in order
e_tot
e_cas
Fci vecs 
mo_coeff
mo_energy'''
mo = casscf.sort_mo([17,20,21,22,23,24])# the numbers are from the index of pi orbitals 
casscf.kernel(mo) 


(np.float64(-230.23764159131332),
 np.float64(-6.486404475722338),
 FCIvector([[ 9.15822691e-01,  0.00000000e+00,  0.00000000e+00,
              0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
              4.40113036e-05,  8.33026859e-02,  0.00000000e+00,
              0.00000000e+00,  0.00000000e+00,  6.37154263e-05,
              0.00000000e+00,  0.00000000e+00, -2.19303208e-02,
              0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
             -2.18393756e-02,  0.00000000e+00],
            [ 0.00000000e+00, -1.74296565e-01,  0.00000000e+00,
              0.00000000e+00,  0.00000000e+00, -3.25749452e-02,
              0.00000000e+00,  0.00000000e+00, -9.10854144e-03,
              0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
              5.60028729e-02, -9.32231305e-03,  0.00000000e+00,
              0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
              0.00000000e+00,  7.68936656e-03],
            [ 0.00000000e+00,  0.00000000e+00, -2.57148150e-02,
     

In [8]:
casscf.fcisolver.max_cycle = 3000
casscf.canonicalize(mo_coeff=None, ci=None, eris=None, sort=False, cas_natorb=False, casdm1=None, verbose=3, with_meta_lowdin=False);

filename = f"{molecule}_Ci_coeffs.txt"

with open(filename, 'w', encoding='utf-8') as f:
    
    f.write("det-alpha,     det-beta,    CI coefficients\n")
    
    
    for c, ia, ib in casscf.fcisolver.large_ci(casscf.ci, n_orb, n_elec, tol=.000000001, return_strs=False):
        # Satır formatını oluşturalım
        line = f" {str(ia):<12} {str(ib):<12} {c:.15f}\n"
        
        #write in file 
        f.write(line)
        #show in terminal 
        print(line.strip())

[0 1 2]      [0 1 2]      0.915822690751131
[0 1 2]      [1 2 4]      0.000044011303554
[0 1 2]      [0 3 4]      0.083302685877998
[0 1 2]      [0 2 5]      0.000063715426261
[0 1 2]      [1 3 5]      -0.021930320817433
[0 1 2]      [2 4 5]      -0.021839375581749
[0 1 3]      [0 1 3]      -0.174296565475269
[0 1 3]      [0 2 4]      -0.032574945183041
[0 1 3]      [1 3 4]      -0.009108541442183
[0 1 3]      [1 2 5]      0.056002872940942
[0 1 3]      [0 3 5]      -0.009322313048969
[0 1 3]      [3 4 5]      0.007689366555244
[0 2 3]      [0 2 3]      -0.025714814956912
[0 2 3]      [0 1 4]      -0.115877858109221
[0 2 3]      [2 3 4]      0.009082371396948
[0 2 3]      [0 4 5]      0.009323836433638
[1 2 3]      [1 2 3]      -0.044391999722031
[1 2 3]      [0 1 5]      0.077933249676710
[1 2 3]      [2 3 5]      -0.005640385943568
[1 2 3]      [1 4 5]      -0.005661832515369
[0 1 4]      [0 2 3]      -0.115877858109221
[0 1 4]      [0 1 4]      -0.025712821901623
[0 1 4]      [2 3 4

In [11]:
#to check the choosen orbitals are pi orbitals printing molden file and check their shapes 
from pyscf.tools import molden

molden.from_mo(mol, f"{molecule}.molden" ,mf.mo_coeff,ene= mf.mo_energy,occ= mf.mo_occ)
